# EMI Pre-Compliance Analysis — Cardiac Biplane (Standby)

**Test date:** 18 May 2026  
**Detector:** QP, 120 kHz IF  
**Antenna:** Schwarzbeck VULB 9162 (TRILOG broadband, 30 MHz – 7 GHz)  
**Scan range:** 30 MHz – 1 GHz  
**Measurement distance:** 1–3 m (pre-compliance, in-situ)

This notebook overlays the radiated-emissions traces taken at three antenna positions, highlights peak emitters, and compares them against the **CISPR 11** limit lines (configurable Class A / Class B, normalised to 10 m).

---

## 1. Configuration

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.signal import find_peaks
import plotly.graph_objects as go

# --- User-tunable parameters ----------------------------------------------
DATA_ROOT      = Path('data/18 May')
CISPR_CLASS    = 'B'        # 'A' or 'B'
MEAS_DISTANCE  = 3.0        # metres at which traces were captured (worst-case distance)
LIMIT_DISTANCE = 10.0       # metres the CISPR 11 limit is defined at
PEAK_PROMINENCE = 6.0       # dB — minimum prominence for a peak to be flagged
MIN_PEAK_SEP_MHZ = 2.0      # cluster peaks closer than this; keep the strongest
TOP_N_PEAKS    = 12         # how many strongest peaks per trace to list
# --------------------------------------------------------------------------

# Distance correction: E ∝ 1/d  →  E(d_limit) = E(d_meas) + 20·log10(d_meas / d_limit)
DIST_CORR_DB = 20.0 * np.log10(MEAS_DISTANCE / LIMIT_DISTANCE)
print(f'Distance correction applied to measurements: {DIST_CORR_DB:+.2f} dB '
      f'(from {MEAS_DISTANCE} m to {LIMIT_DISTANCE} m reference)')

Distance correction applied to measurements: -10.46 dB (from 3.0 m to 10.0 m reference)


## 2. Load measurement traces

In [2]:
POSITIONS = {
    'Antenna at center':
        DATA_ROOT / 'Antenna on center' / 'Antenna in center_Standby_1to3m.txt',
    'Antenna on cabinets side':
        DATA_ROOT / 'Antenna on Cabinets side' / 'Antenna in cabinets side_Standby_1to3m.txt',
    'Antenna away from cabinets':
        DATA_ROOT / 'Antenna away from cabinets corner' /
        'Antenna on other side away from cabinets_Standby_1to3m.txt',
}

def load_trace(path: Path) -> pd.DataFrame:
    """Read a Rohde & Schwarz EMI scan file (% header, 3 columns)."""
    df = pd.read_csv(
        path, sep=r'\s+', comment='%',
        names=['freq_hz', 'mag_raw_dbuv', 'mag_field_dbuvm'],
        engine='python', skip_blank_lines=True,
    ).dropna().reset_index(drop=True)
    df['freq_mhz'] = df['freq_hz'] / 1e6
    df['field_corr_dbuvm'] = df['mag_field_dbuvm'] + DIST_CORR_DB
    return df

traces = {name: load_trace(p) for name, p in POSITIONS.items()}
for name, df in traces.items():
    print(f'{name:30s}  {len(df):>5d} pts   '
          f'{df.freq_mhz.min():7.2f} – {df.freq_mhz.max():7.2f} MHz   '
          f'peak {df.field_corr_dbuvm.max():5.1f} dBµV/m')

Antenna at center               19867 pts     30.00 – 1000.00 MHz   peak  47.9 dBµV/m
Antenna on cabinets side        19867 pts     30.00 – 1000.00 MHz   peak  51.6 dBµV/m
Antenna away from cabinets      19867 pts     30.00 – 1000.00 MHz   peak  43.0 dBµV/m


## 3. CISPR 11 limit line (Group 1, radiated, normalised to 10 m)

In [3]:
# CISPR 11, Group 1, radiated QP limits at 10 m measurement distance
CISPR11_LIMITS = {
    'A': [(30.0, 230.0, 40.0), (230.0, 1000.0, 47.0)],
    'B': [(30.0, 230.0, 30.0), (230.0, 1000.0, 37.0)],
}

def limit_at(freq_mhz: np.ndarray, cls: str = CISPR_CLASS) -> np.ndarray:
    out = np.full_like(freq_mhz, np.nan, dtype=float)
    for f_lo, f_hi, lvl in CISPR11_LIMITS[cls]:
        mask = (freq_mhz >= f_lo) & (freq_mhz <= f_hi)
        out[mask] = lvl
    return out

# Build a stepped limit trace for plotting (with the 230 MHz step)
lim_f = np.array([30, 230, 230, 1000], dtype=float)
lim_v = np.array([s[2] for s in CISPR11_LIMITS[CISPR_CLASS] for _ in (0, 1)], dtype=float)
print(f'Using CISPR 11 Group 1, Class {CISPR_CLASS} limits at {LIMIT_DISTANCE:.0f} m:')
for f_lo, f_hi, lvl in CISPR11_LIMITS[CISPR_CLASS]:
    print(f'  {f_lo:>6.1f} – {f_hi:>6.1f} MHz : {lvl:.0f} dBµV/m (QP)')

Using CISPR 11 Group 1, Class B limits at 10 m:
    30.0 –  230.0 MHz : 30 dBµV/m (QP)
   230.0 – 1000.0 MHz : 37 dBµV/m (QP)


## 4. Peak detection

Peaks are required to be at least `PEAK_PROMINENCE` dB above the local baseline and separated by at least `MIN_PEAK_SEP_MHZ` MHz, so densely sampled humps collapse to a single entry.

In [4]:
def detect_peaks(df: pd.DataFrame,
                 prominence_db: float = PEAK_PROMINENCE,
                 min_sep_mhz: float = MIN_PEAK_SEP_MHZ) -> pd.DataFrame:
    y = df['field_corr_dbuvm'].to_numpy()
    f = df['freq_mhz'].to_numpy()
    step = float(np.median(np.diff(f)))
    distance = max(1, int(round(min_sep_mhz / step)))
    idx, props = find_peaks(y, prominence=prominence_db, distance=distance)
    if len(idx) == 0:
        return pd.DataFrame(columns=['freq_mhz', 'level_dbuvm',
                                     'prominence_db', 'limit_dbuvm', 'margin_db'])
    peaks = pd.DataFrame({
        'freq_mhz':      f[idx],
        'level_dbuvm':   y[idx],
        'prominence_db': props['prominences'],
    })
    peaks['limit_dbuvm'] = limit_at(peaks['freq_mhz'].to_numpy())
    # margin = limit − level  (positive = pass, negative = exceeds limit)
    peaks['margin_db'] = peaks['limit_dbuvm'] - peaks['level_dbuvm']
    return peaks.sort_values('level_dbuvm', ascending=False).reset_index(drop=True)

peak_tables = {name: detect_peaks(df) for name, df in traces.items()}
for name, tbl in peak_tables.items():
    n_fail = (tbl['margin_db'] < 0).sum()
    print(f'{name:30s}  {len(tbl):>3d} peaks  ({n_fail} above limit)')

Antenna at center                15 peaks  (6 above limit)
Antenna on cabinets side         15 peaks  (7 above limit)
Antenna away from cabinets       16 peaks  (6 above limit)


## 5. Overlay plot with peak markers

Interactive — hover to read frequency/level, click legend entries to isolate a trace, drag to zoom.

In [5]:
COLORS = {
    'Antenna at center':           '#1f77b4',
    'Antenna on cabinets side':    '#d62728',
    'Antenna away from cabinets':  '#2ca02c',
}

fig = go.Figure()

# CISPR 11 limit (drawn first so it sits behind traces)
fig.add_trace(go.Scatter(
    x=lim_f, y=lim_v, mode='lines',
    name=f'CISPR 11 Class {CISPR_CLASS} @ {LIMIT_DISTANCE:.0f} m (QP)',
    line=dict(color='black', dash='dash', width=2),
    hovertemplate='Limit: %{y:.0f} dBµV/m<extra></extra>',
))

for name, df in traces.items():
    color = COLORS[name]
    fig.add_trace(go.Scatter(
        x=df['freq_mhz'], y=df['field_corr_dbuvm'], mode='lines',
        name=name, line=dict(color=color, width=1.2),
        hovertemplate='%{x:.2f} MHz<br>%{y:.2f} dBµV/m<extra>' + name + '</extra>',
    ))
    pk = peak_tables[name].head(TOP_N_PEAKS)
    fig.add_trace(go.Scatter(
        x=pk['freq_mhz'], y=pk['level_dbuvm'],
        mode='markers+text',
        marker=dict(color=color, size=9, symbol='triangle-down',
                    line=dict(color='black', width=0.6)),
        text=[f'{f:.0f}' for f in pk['freq_mhz']],
        textposition='top center', textfont=dict(size=9, color=color),
        name=f'{name} — peaks', showlegend=False,
        hovertemplate='%{x:.2f} MHz<br>%{y:.2f} dBµV/m<extra>peak</extra>',
    ))

fig.update_xaxes(
    title='Frequency (MHz)', type='log',
    tickvals=[30, 50, 100, 200, 300, 500, 700, 1000],
    range=[np.log10(30), np.log10(1000)],
    showgrid=True, gridcolor='lightgrey',
)
fig.update_yaxes(title='Field strength (dBµV/m, QP)',
                 showgrid=True, gridcolor='lightgrey')
fig.update_layout(
    title=f'Radiated emissions — Standby — three antenna positions '
          f'(normalised to {LIMIT_DISTANCE:.0f} m)',
    template='plotly_white', height=620,
    legend=dict(orientation='h', y=-0.18, x=0.0),
    margin=dict(l=70, r=30, t=70, b=80),
)
fig.show()

## 6. Worst-case envelope across the three positions

In [6]:
ref = next(iter(traces.values()))
f_grid = ref['freq_mhz'].to_numpy()

stacked = {name: np.interp(f_grid, df['freq_mhz'], df['field_corr_dbuvm'])
           for name, df in traces.items()}
env = np.max(np.vstack(list(stacked.values())), axis=0)
env_df = pd.DataFrame({'freq_mhz': f_grid, 'field_corr_dbuvm': env})
env_peaks = detect_peaks(env_df)

fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=lim_f, y=lim_v, mode='lines',
    name=f'CISPR 11 Class {CISPR_CLASS} @ {LIMIT_DISTANCE:.0f} m',
    line=dict(color='black', dash='dash', width=2),
))
fig2.add_trace(go.Scatter(
    x=f_grid, y=env, mode='lines',
    name='Worst-case envelope (max of 3 positions)',
    line=dict(color='#ff7f0e', width=1.4),
    hovertemplate='%{x:.2f} MHz<br>%{y:.2f} dBµV/m<extra></extra>',
))
pk = env_peaks.head(TOP_N_PEAKS)
fig2.add_trace(go.Scatter(
    x=pk['freq_mhz'], y=pk['level_dbuvm'],
    mode='markers+text',
    marker=dict(color='#ff7f0e', size=10, symbol='triangle-down',
                line=dict(color='black', width=0.6)),
    text=[f'{f:.0f} MHz' for f in pk['freq_mhz']],
    textposition='top center', textfont=dict(size=9),
    name='Envelope peaks', showlegend=False,
))
fig2.update_xaxes(title='Frequency (MHz)', type='log',
                  tickvals=[30, 50, 100, 200, 300, 500, 700, 1000],
                  range=[np.log10(30), np.log10(1000)],
                  showgrid=True, gridcolor='lightgrey')
fig2.update_yaxes(title='Field strength (dBµV/m, QP)',
                  showgrid=True, gridcolor='lightgrey')
fig2.update_layout(
    title=f'Worst-case envelope vs. CISPR 11 Class {CISPR_CLASS}',
    template='plotly_white', height=560,
    legend=dict(orientation='h', y=-0.2, x=0.0),
    margin=dict(l=70, r=30, t=70, b=80),
)
fig2.show()

## 7. Peak table (top emitters per position)

*Margin* = limit − measured level. **Positive ⇒ pass, negative ⇒ exceeds limit.**

In [7]:
def format_peak_table(tbl: pd.DataFrame, position: str) -> pd.DataFrame:
    out = tbl.head(TOP_N_PEAKS).copy()
    out.insert(0, 'Position', position)
    out['Status'] = np.where(out['margin_db'] >= 0, 'PASS', 'FAIL')
    out = out[['Position', 'freq_mhz', 'level_dbuvm', 'limit_dbuvm',
               'margin_db', 'Status']]
    out.columns = ['Position', 'Freq (MHz)', 'Level (dBµV/m)',
                   f'Limit Cl.{CISPR_CLASS}', 'Margin (dB)', 'Status']
    return out

all_peaks = pd.concat(
    [format_peak_table(tbl, name) for name, tbl in peak_tables.items()],
    ignore_index=True,
)

def _style(df):
    return (df.style
              .format({'Freq (MHz)':              '{:.2f}',
                       'Level (dBµV/m)':         '{:.1f}',
                       f'Limit Cl.{CISPR_CLASS}': '{:.0f}',
                       'Margin (dB)':            '{:+.1f}'})
              .map(lambda v: ('color:#b00020;font-weight:600'
                              if isinstance(v, str) and v == 'FAIL'
                              else 'color:#1b7f3a'),
                   subset=['Status'])
              .background_gradient(subset=['Margin (dB)'],
                                   cmap='RdYlGn', vmin=-10, vmax=20))

_style(all_peaks)

,Position,Freq (MHz),Level (dBµV/m),Limit Cl.B,Margin (dB),Status
0,Antenna at center,898.65,47.9,37,-10.9,FAIL
1,Antenna at center,857.44,42.9,37,-5.9,FAIL
2,Antenna at center,861.01,42.7,37,-5.7,FAIL
3,Antenna at center,855.00,42.3,37,-5.3,FAIL
4,Antenna at center,852.90,39.3,37,-2.3,FAIL
5,Antenna at center,39.91,32.7,30,-2.7,FAIL
6,Antenna at center,514.38,29.6,37,+7.4,PASS
7,Antenna at center,494.50,28.8,37,+8.2,PASS
8,Antenna at center,490.01,28.7,37,+8.3,PASS
9,Antenna at center,162.52,28.7,30,+1.3,PASS


### Worst-case envelope peaks

In [8]:
_style(format_peak_table(env_peaks, 'Worst-case envelope'))

,Position,Freq (MHz),Level (dBµV/m),Limit Cl.B,Margin (dB),Status
0,Worst-case envelope,861.05,51.6,37,-14.6,FAIL
1,Worst-case envelope,853.44,49.5,37,-12.5,FAIL
2,Worst-case envelope,898.65,47.9,37,-10.9,FAIL
3,Worst-case envelope,858.03,45.8,37,-8.8,FAIL
4,Worst-case envelope,162.52,34.4,30,-4.4,FAIL
5,Worst-case envelope,39.91,32.7,30,-2.7,FAIL
6,Worst-case envelope,262.52,30.9,37,+6.1,PASS
7,Worst-case envelope,700.02,30.7,37,+6.3,PASS
8,Worst-case envelope,487.81,30.1,37,+6.9,PASS
9,Worst-case envelope,514.38,29.6,37,+7.4,PASS


## 8. Export the tables and standalone HTML plots

In [9]:
out_dir = Path('reports'); out_dir.mkdir(exist_ok=True)
all_peaks.to_csv(out_dir / 'peaks_per_position.csv', index=False)
format_peak_table(env_peaks, 'Worst-case envelope').to_csv(
    out_dir / 'peaks_envelope.csv', index=False)
fig.write_html(out_dir / 'overlay_three_positions.html', include_plotlyjs='cdn')
fig2.write_html(out_dir / 'envelope_vs_cispr11.html', include_plotlyjs='cdn')
print('Wrote:')
for p in sorted(out_dir.iterdir()):
    print('  ', p)

Wrote:
   reports\envelope_vs_cispr11.html
   reports\overlay_three_positions.html
   reports\peaks_envelope.csv
   reports\peaks_per_position.csv


---
### Notes for the presentation

* Limits used are **CISPR 11, Group 1, radiated QP at 10 m**. Switch `CISPR_CLASS` to `'A'` in section 1 for the industrial / professional-healthcare limit.
* Distance correction assumes far-field 1/r propagation. Below ~230 MHz at 3 m this is an approximation — treat the margin as indicative, not as a formal compliance result.
* Antenna polarisation, height scan and turntable rotation are **not** part of this pre-compliance sweep; expect a few dB of additional uncertainty when comparing to a 10 m chamber result.
* Tune `PEAK_PROMINENCE` (higher = fewer/cleaner markers) and `MIN_PEAK_SEP_MHZ` (wider = merge nearby harmonics) for your audience.
* The `reports/` folder contains exportable HTML versions of both plots and CSV peak tables — open them directly in a browser or drop them into a slide deck.